# InduSense — Sprint 1 : cartes d'identité des fichiers CSV

Pour chaque fichier CSV de `datas/`, ce notebook produit une carte d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md` contenant :

- la source, la taille, l'heure de début et de fin de traitement, le charset,
- le nombre de lignes et de colonnes,
- la liste des colonnes avec leur type inféré,
- min / max (et mean / median pour les numériques) des colonnes numériques et dates.

In [1]:
import csv
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

DATA_DIR = Path("datas")
ARTIFACTS_DIR = Path("artifacts/ingestion/identities")
CSV_FILES = [
    DATA_DIR / "releves_incidents.csv",
    DATA_DIR / "telemetry.csv",
]

## 1. Lecture du fichier : taille, charset et séparateur

Le charset est déterminé en essayant de décoder le fichier (ASCII, puis UTF-8, avec ou sans BOM).
À défaut, on considère le fichier en ISO-8859-1 (Latin-1), qui accepte n'importe quel octet.

Le séparateur n'est pas toujours une virgule : il est détecté par `csv.Sniffer` parmi `,` `;` `\t` `|`.

In [2]:
def format_filesize(size_bytes: int) -> str:
    """Formate une taille en octets : 9999.99 [octets | Ko | Mo | Go]."""
    size = float(size_bytes)
    for unit in ("octets", "Ko", "Mo"):
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} Go"


def detect_charset(path: Path) -> tuple[str, str]:
    """Retourne (libellé du charset, encodage Python à utiliser pour lire le fichier)."""
    raw = path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        return "UTF-8 (BOM)", "utf-8-sig"
    for label, encoding in (("ASCII", "ascii"), ("UTF-8", "utf-8")):
        try:
            raw.decode(encoding)
            return label, encoding
        except UnicodeDecodeError:
            pass
    return "ISO-8859-1 (Latin-1)", "latin-1"


def detect_separator(path: Path, encoding: str) -> str:
    """Détecte le séparateur du CSV à partir d'un échantillon du début du fichier."""
    with path.open(encoding=encoding, newline="") as f:
        sample = f.read(64 * 1024)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","


for path in CSV_FILES:
    charset, encoding = detect_charset(path)
    print(f"{path.name}: {format_filesize(path.stat().st_size)}, {charset}, "
          f"séparateur {detect_separator(path, encoding)!r}")

releves_incidents.csv: 134.64 Ko, UTF-8, séparateur ','
telemetry.csv: 7.94 Mo, ASCII, séparateur ','


## 2. Inférence du type des colonnes

| Type | Règle |
| --- | --- |
| `IDENTITY` | nom `id` / `*_id`, ou valeurs de la forme `MACH-01`, `INC-000001`, `OP1002` |
| `BOOLEAN` | valeurs uniquement 0 / 1 (ou booléens) |
| `INTEGER` / `FLOAT` | colonne numérique |
| `DATE` / `DATETIME` | valeurs au format ISO 8601 (`2025-06-01`, `2025-06-01 05:42:00`) |
| `TIME` | valeurs `HH:MM` ou `HH:MM:SS` |
| `CATEGORY` | texte avec au plus 20 valeurs distinctes |
| `TEXT` | autre texte |

In [3]:
IDENTITY_PATTERN = r"[A-Za-z]+-?\d+"
TIME_PATTERN = r"\d{2}:\d{2}(:\d{2})?"
MAX_CATEGORIES = 20

NUMERIC_TYPES = {"INTEGER", "FLOAT"}
TEMPORAL_TYPES = {"DATE", "DATETIME", "TIME"}


def infer_column_type(name: str, series: pd.Series) -> str:
    """Infère le type métier d'une colonne."""
    values = series.dropna()
    if values.empty:
        return "EMPTY"
    if name.lower() == "id" or name.lower().endswith("_id"):
        return "IDENTITY"
    if pd.api.types.is_bool_dtype(values):
        return "BOOLEAN"
    if pd.api.types.is_numeric_dtype(values):
        if set(values.unique()) <= {0, 1}:
            return "BOOLEAN"
        return "INTEGER" if pd.api.types.is_integer_dtype(values) else "FLOAT"

    text = values.astype(str).str.strip()
    if text.str.fullmatch(IDENTITY_PATTERN).all():
        return "IDENTITY"
    if text.str.fullmatch(TIME_PATTERN).all():
        return "TIME"
    dates = pd.to_datetime(text, format="ISO8601", errors="coerce")
    if dates.notna().all():
        is_date_only = (dates == dates.dt.normalize()).all() and text.str.len().max() <= 10
        return "DATE" if is_date_only else "DATETIME"
    if text.nunique() <= MAX_CATEGORIES:
        return "CATEGORY"
    return "TEXT"

## 3. Statistiques des colonnes numériques et dates

Min et max pour toutes ces colonnes ; mean et median uniquement pour les numériques.

In [4]:
def format_value(value) -> str:
    """Formate une valeur pour l'affichage dans la carte d'identité."""
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.3f}"
    return str(value)


def column_stats(series: pd.Series, col_type: str) -> dict:
    """Calcule min / max (et mean / median pour les numériques) d'une colonne."""
    values = series.dropna()
    if col_type in NUMERIC_TYPES:
        return {
            "min": values.min().item(),
            "max": values.max().item(),
            "mean": float(values.mean()),
            "median": float(values.median()),
        }
    if col_type == "TIME":
        # Format HH:MM zéro-paddé : l'ordre alphabétique est l'ordre chronologique
        text = values.astype(str).str.strip()
        return {"min": text.min(), "max": text.max(), "mean": None, "median": None}
    dates = pd.to_datetime(values.astype(str).str.strip(), format="ISO8601")
    date_format = "%Y-%m-%d" if col_type == "DATE" else "%Y-%m-%d %H:%M:%S"
    return {
        "min": dates.min().strftime(date_format),
        "max": dates.max().strftime(date_format),
        "mean": None,
        "median": None,
    }

## 4. Construction et écriture de la carte d'identité

In [5]:
def build_identity(path: Path) -> dict:
    """Analyse un fichier CSV et retourne sa carte d'identité sous forme de dictionnaire."""
    begin = datetime.now()
    charset, encoding = detect_charset(path)
    separator = detect_separator(path, encoding)
    df = pd.read_csv(path, sep=separator, encoding=encoding)

    types = {col: infer_column_type(col, df[col]) for col in df.columns}
    stats = {
        col: column_stats(df[col], col_type)
        for col, col_type in types.items()
        if col_type in NUMERIC_TYPES | TEMPORAL_TYPES
    }
    return {
        "source": path.resolve(),
        "filesize": format_filesize(path.stat().st_size),
        "begin": begin,
        "end": datetime.now(),
        "charset": charset,
        "rows": len(df),
        "cols": len(df.columns),
        "types": types,
        "stats": stats,
    }


def render_identity(identity: dict) -> str:
    """Met en forme la carte d'identité en Markdown."""
    lines = [
        f"# Carte d'identité : {identity['source'].name}",
        "",
        f"- Source: {identity['source']}",
        f"- Filesize: {identity['filesize']}",
        f"- ProcessBeginAt: {identity['begin']:%H:%M:%S}",
        f"- ProcessEndAt: {identity['end']:%H:%M:%S}",
        f"- Charset: {identity['charset']}",
        f"- Rows Number: {identity['rows']}",
        f"- Cols Number: {identity['cols']}",
        "",
        "## Column list",
        "",
        "| Colname | Inferred Type |",
        "| --- | --- |",
    ]
    lines += [f"| {col} | {col_type} |" for col, col_type in identity["types"].items()]
    lines += [
        "",
        "## Numeric and date columns",
        "",
        "| Colname | Min value | Max value | Mean value | Median value |",
        "| --- | --- | --- | --- | --- |",
    ]
    lines += [
        f"| {col} | {format_value(s['min'])} | {format_value(s['max'])} "
        f"| {format_value(s['mean'])} | {format_value(s['median'])} |"
        for col, s in identity["stats"].items()
    ]
    return "\n".join(lines) + "\n"


def write_identity(identity: dict, output_root: Path = ARTIFACTS_DIR) -> Path:
    """Écrit la carte dans <output_root>/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md."""
    begin = identity["begin"]
    output_dir = output_root / f"{begin:%Y%m%d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{identity['source'].stem}_{begin:%H%M%S}.md"
    output_path.write_text(render_identity(identity), encoding="utf-8")
    return output_path

## 5. Génération des cartes d'identité

In [6]:
for path in CSV_FILES:
    identity = build_identity(path)
    output_path = write_identity(identity)
    print(f"Carte générée : {output_path}")
    display(Markdown(render_identity(identity)))

Carte générée : artifacts/ingestion/identities/20261120/releves_incidents_145215.md


# Carte d'identité : releves_incidents.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/releves_incidents.csv
- Filesize: 134.64 Ko
- ProcessBeginAt: 14:52:15
- ProcessEndAt: 14:52:15
- Charset: UTF-8
- Rows Number: 1245
- Cols Number: 18

## Column list

| Colname | Inferred Type |
| --- | --- |
| incident_id | IDENTITY |
| date | DATE |
| time | TIME |
| operator_name | CATEGORY |
| machine_id | IDENTITY |
| severity | INTEGER |
| operator_badge | IDENTITY |
| comment | TEXT |
| shift | CATEGORY |
| type_surchauffe | BOOLEAN |
| type_baisse_pression | BOOLEAN |
| type_vibration | BOOLEAN |
| type_bruit_mecanique | BOOLEAN |
| type_surconsommation | BOOLEAN |
| type_blocage_mecanique | BOOLEAN |
| type_alarme_capteur | BOOLEAN |
| type_arret_urgence | BOOLEAN |
| type_defaut_qualite | BOOLEAN |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| date | 2025-06-01 | 2026-06-08 | - | - |
| time | 00:06 | 23:55 | - | - |
| severity | 2 | 5 | 2.925 | 3.000 |


Carte générée : artifacts/ingestion/identities/20261120/telemetry_145215.md


# Carte d'identité : telemetry.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/telemetry.csv
- Filesize: 7.94 Mo
- ProcessBeginAt: 14:52:15
- ProcessEndAt: 14:52:15
- Charset: ASCII
- Rows Number: 135626
- Cols Number: 7

## Column list

| Colname | Inferred Type |
| --- | --- |
| machine_id | IDENTITY |
| timestamp | DATETIME |
| temperature_c | FLOAT |
| pressure_bar | FLOAT |
| voltage_mean_v | FLOAT |
| rotation_mean_rpm | FLOAT |
| pieces_produced | INTEGER |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| timestamp | 2025-06-01 00:00:00 | 2026-06-08 23:00:00 | - | - |
| temperature_c | 32.000 | 80.000 | 48.183 | 48.055 |
| pressure_bar | 159.982 | 215.814 | 199.770 | 199.866 |
| voltage_mean_v | 220.900 | 242.000 | 227.631 | 227.420 |
| rotation_mean_rpm | 1100.000 | 1900.000 | 1589.205 | 1590.370 |
| pieces_produced | 0 | 114 | 49.533 | 49.000 |
